# Homogeneous Transformation for Planar and Spatial Mechanisms
**Python Robotics Toolbox — Experiment 2**

This notebook covers all transformation tasks from Part A (SE2) through Part D (Challenge). Each task is implemented using the `spatialmath` library from the Python Robotics Toolbox. Frames are visualized using `matplotlib`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from spatialmath import SE2, SE3, SO3
from spatialmath.base import trplot2, trplot

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

# Helper: draw a 2D coordinate frame given an SE2 transform
def plot_frame_2d(T, ax, label='', scale=0.5, color_x='r', color_y='g'):
    origin = T.t
    R = T.A[:2, :2]
    for vec, col, lbl in zip([np.array([scale,0]), np.array([0,scale])],
                              [color_x, color_y], ['X','Y']):
        tip = origin + R @ vec
        ax.annotate('', xy=tip, xytext=origin,
                    arrowprops=dict(arrowstyle='->', color=col, lw=1.5))
        ax.text(*tip, f' {lbl}', color=col, fontsize=8)
    if label:
        ax.text(origin[0], origin[1]-0.15, label, fontsize=8, ha='center')

# Helper: draw a 3D coordinate frame given an SE3 transform
def plot_frame_3d(T, ax, label='', scale=0.5):
    origin = T.t
    R = T.A[:3, :3]
    colors = ['r','g','b']
    labels = ['X','Y','Z']
    for i in range(3):
        tip = origin + scale * R[:, i]
        ax.quiver(*origin, *(tip - origin), color=colors[i], arrow_length_ratio=0.2)
        ax.text(*tip, f' {labels[i]}', color=colors[i], fontsize=8)
    if label:
        ax.text(*origin, f'  {label}', fontsize=8)

print('Setup complete.')

---
## Part A — Planar Mechanism (SE2)

---
### Task 1 — Planar Translation

A planar link at the origin is translated by (4, 3). We use `SE2(x, y)` to construct the translation matrix, then repeat for three more translation vectors. Both the original and translated frames are plotted.

In [ ]:
translations = [(4, 3), (2, 5), (5, 1), (-3, 4)]

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()

T_origin = SE2(0, 0)  # reference frame at origin

for idx, (tx, ty) in enumerate(translations):
    T = SE2(tx, ty)
    print(f"\n--- Translation ({tx}, {ty}) ---")
    print("Homogeneous Transformation Matrix:")
    print(np.round(T.A, 4))
    print(f"Final position: X = {T.t[0]}, Y = {T.t[1]}")

    ax = axes[idx]
    plot_frame_2d(T_origin, ax, label='Origin')
    plot_frame_2d(T, ax, label=f'T({tx},{ty})', color_x='darkred', color_y='darkgreen')
    ax.set_xlim(min(-1, tx-1), max(2, tx+1))
    ax.set_ylim(min(-1, ty-1), max(2, ty+1))
    ax.set_aspect('equal')
    ax.grid(True, ls='--', alpha=0.5)
    ax.set_title(f'Translation ({tx}, {ty})')
    ax.set_xlabel('X'); ax.set_ylabel('Y')

plt.suptitle('Task 1 — Planar Translation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
### Task 2 — Planar Rotation

A planar link aligned with the X-axis is rotated counter-clockwise by various angles. `SE2.Rot(theta)` constructs a pure rotation. The rotation matrix is a 3×3 homogeneous matrix where the top-left 2×2 block is the standard 2D rotation matrix.

In [ ]:
angles_deg = [60, 30, 90, 120, 180]

fig, axes = plt.subplots(1, 5, figsize=(15, 4))
T_origin = SE2(0, 0)

for idx, angle in enumerate(angles_deg):
    T = SE2.Rot(np.deg2rad(angle))
    print(f"\n--- Rotation {angle}° ---")
    print("Transformation Matrix:")
    print(np.round(T.A, 4))
    print(f"Orientation: {angle}°")

    ax = axes[idx]
    plot_frame_2d(T_origin, ax, label='Origin')
    plot_frame_2d(T, ax, label=f'{angle}°', color_x='orange', color_y='purple')
    ax.set_xlim(-1, 1.5); ax.set_ylim(-1, 1.5)
    ax.set_aspect('equal')
    ax.grid(True, ls='--', alpha=0.5)
    ax.set_title(f'Rotation {angle}°')
    ax.set_xlabel('X'); ax.set_ylabel('Y')

plt.suptitle('Task 2 — Planar Rotation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
### Task 3 — Planar Translation and Rotation

Here we combine a translation and a rotation. The combined transform is `T = SE2(x,y) * SE2.Rot(theta)` — translation first, then rotation. This is equivalent to multiplying the individual homogeneous matrices.

In [ ]:
cases = [(3, 2, 45), (5, 1, 60), (2, 4, 90)]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
T_origin = SE2(0, 0)

for idx, (tx, ty, angle) in enumerate(cases):
    T_trans = SE2(tx, ty)                      # translation matrix
    T_rot   = SE2.Rot(np.deg2rad(angle))       # rotation matrix
    T_combined = T_trans * T_rot               # combined: translate then rotate

    print(f"\n--- Case {idx+1}: T=({tx},{ty}), R={angle}° ---")
    print(f"Translation Matrix:\n{np.round(T_trans.A, 4)}")
    print(f"Rotation Matrix:\n{np.round(T_rot.A, 4)}")
    print(f"Combined Matrix:\n{np.round(T_combined.A, 4)}")
    print(f"Final position: X={T_combined.t[0]:.4f}, Y={T_combined.t[1]:.4f}")
    print(f"Final orientation: {angle}°")

    ax = axes[idx]
    plot_frame_2d(T_origin, ax, label='Origin')
    plot_frame_2d(T_combined, ax, label=f'T+R', color_x='darkred', color_y='darkgreen')
    ax.set_xlim(-1, tx+1.5); ax.set_ylim(-1, ty+1.5)
    ax.set_aspect('equal')
    ax.grid(True, ls='--', alpha=0.5)
    ax.set_title(f'Case {idx+1}: ({tx},{ty}), {angle}°')
    ax.set_xlabel('X'); ax.set_ylabel('Y')

plt.suptitle('Task 3 — Planar Translation + Rotation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
### Task 4 — Effect of Transformation Order

Matrix multiplication is **not commutative** for rigid body transforms. Here we compare:
- **Case A**: Translation(4,2) × Rotation(90°) — translate first, then rotate
- **Case B**: Rotation(90°) × Translation(4,2) — rotate first, then translate

The final positions will differ because the translation in Case B is applied in the rotated frame.

In [ ]:
T_trans = SE2(4, 2)
T_rot   = SE2.Rot(np.deg2rad(90))

T_A = T_trans * T_rot   # Case A: Translation then Rotation
T_B = T_rot * T_trans   # Case B: Rotation then Translation

print("Case A (Translation → Rotation):")
print(np.round(T_A.A, 4))
print(f"Final position: X={T_A.t[0]:.4f}, Y={T_A.t[1]:.4f}")

print("\nCase B (Rotation → Translation):")
print(np.round(T_B.A, 4))
print(f"Final position: X={T_B.t[0]:.4f}, Y={T_B.t[1]:.4f}")

print("\nAre matrices equal?", np.allclose(T_A.A, T_B.A))

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
T_origin = SE2(0, 0)
titles = ['Case A: Trans → Rot', 'Case B: Rot → Trans']
for ax, T, title in zip(axes, [T_A, T_B], titles):
    plot_frame_2d(T_origin, ax, label='Origin')
    plot_frame_2d(T, ax, label='Final', color_x='darkred', color_y='darkgreen')
    ax.set_xlim(-3, 6); ax.set_ylim(-3, 6)
    ax.set_aspect('equal')
    ax.grid(True, ls='--', alpha=0.5)
    ax.set_title(title)
    ax.set_xlabel('X'); ax.set_ylabel('Y')

plt.suptitle('Task 4 — Effect of Transformation Order', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
### Task 5 — Sequential Planar Transformation

The robot applies four transforms in order: Translate(2,1) → Rotate(30°) → Translate(3,2) → Rotate(45°). We chain them using sequential matrix multiplication and then compare what happens if the order is reversed.

In [ ]:
# individual transforms
T1 = SE2(2, 1)                        # translate (2,1)
T2 = SE2.Rot(np.deg2rad(30))          # rotate 30°
T3 = SE2(3, 2)                        # translate (3,2)
T4 = SE2.Rot(np.deg2rad(45))          # rotate 45°

T_seq = T1 * T2 * T3 * T4            # sequential combination

print("Individual matrices:")
for i, T in enumerate([T1, T2, T3, T4], 1):
    print(f"T{i}:\n{np.round(T.A, 4)}")

print("\nFinal Combined Matrix:")
print(np.round(T_seq.A, 4))
print(f"Final X: {T_seq.t[0]:.4f}")
print(f"Final Y: {T_seq.t[1]:.4f}")

# extract orientation angle from rotation matrix
angle_final = np.rad2deg(np.arctan2(T_seq.A[1,0], T_seq.A[0,0]))
print(f"Final orientation: {angle_final:.4f}°")

# reversed order
T_rev = T4 * T3 * T2 * T1
print("\nReversed order matrix:")
print(np.round(T_rev.A, 4))
print(f"Reversed Final X: {T_rev.t[0]:.4f}, Y: {T_rev.t[1]:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
T_origin = SE2(0, 0)
for ax, T, title in zip(axes, [T_seq, T_rev], ['Original Order', 'Reversed Order']):
    plot_frame_2d(T_origin, ax, label='Origin')
    plot_frame_2d(T, ax, label='Final', color_x='darkred', color_y='darkgreen')
    lim = max(abs(T.t[0]), abs(T.t[1])) + 1.5
    ax.set_xlim(-1, lim); ax.set_ylim(-1, lim)
    ax.set_aspect('equal')
    ax.grid(True, ls='--', alpha=0.5)
    ax.set_title(title)
    ax.set_xlabel('X'); ax.set_ylabel('Y')

plt.suptitle('Task 5 — Sequential Planar Transformation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Part B — Spatial Mechanism (SE3)

---
### Task 6 — 3D Translation

The end-effector moves from the origin to a position in 3D space. `SE3(x, y, z)` constructs a pure translation matrix. The last column of the 4×4 matrix gives the position vector.

In [ ]:
cases_3d = [(3,4,2), (5,2,3), (-2,4,5)]

fig = plt.figure(figsize=(13, 4))
T_origin = SE3(0, 0, 0)

for idx, (x, y, z) in enumerate(cases_3d):
    T = SE3(x, y, z)
    print(f"\n--- Translation ({x},{y},{z}) ---")
    print("SE3 Transformation Matrix:")
    print(np.round(T.A, 4))
    print(f"Final position: X={T.t[0]}, Y={T.t[1]}, Z={T.t[2]}")

    ax = fig.add_subplot(1, 3, idx+1, projection='3d')
    plot_frame_3d(T_origin, ax, label='O', scale=0.4)
    plot_frame_3d(T, ax, label='T', scale=0.4)
    lim = max(abs(x), abs(y), abs(z)) + 1
    ax.set_xlim(-1, lim); ax.set_ylim(-1, lim); ax.set_zlim(-1, lim)
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    ax.set_title(f'({x},{y},{z})')

plt.suptitle('Task 6 — 3D Translation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
### Task 7 — Rotation About X, Y and Z Axes

We apply three independent rotations using `SE3.Rx()`, `SE3.Ry()`, `SE3.Rz()`. Each gives a 4×4 homogeneous matrix with the corresponding 3D rotation in the top-left 3×3 block.

In [ ]:
angles_axes = [(45, 'X', SE3.Rx), (60, 'Y', SE3.Ry), (90, 'Z', SE3.Rz)]

fig = plt.figure(figsize=(13, 4))
T_origin = SE3(0, 0, 0)

for idx, (angle, axis, rot_fn) in enumerate(angles_axes):
    T = rot_fn(np.deg2rad(angle))
    print(f"\nRotation {angle}° about {axis}-axis:")
    print(np.round(T.A, 4))

    ax = fig.add_subplot(1, 3, idx+1, projection='3d')
    plot_frame_3d(T_origin, ax, label='O', scale=0.4)
    plot_frame_3d(T, ax, label=f'R{axis}', scale=0.4)
    ax.set_xlim(-1,1); ax.set_ylim(-1,1); ax.set_zlim(-1,1)
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    ax.set_title(f'Rot {angle}° about {axis}')

plt.suptitle('Task 7 — 3D Rotations', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
### Task 8 — 3D Translation Followed by Rotation

The end-effector translates to (2,3,4) and then rotates. We repeat for rotation about all three axes. The combined matrix is `T = T_trans * T_rot`.

In [ ]:
T_trans = SE3(2, 3, 4)
rotations = [(SE3.Rx, 'X'), (SE3.Ry, 'Y'), (SE3.Rz, 'Z')]

fig = plt.figure(figsize=(13, 4))
T_origin = SE3(0, 0, 0)

for idx, (rot_fn, axis) in enumerate(rotations):
    T_rot = rot_fn(np.deg2rad(60))
    T_combined = T_trans * T_rot

    print(f"\n--- Translation (2,3,4) + Rotation 60° about {axis} ---")
    print(f"Translation Matrix:\n{np.round(T_trans.A, 4)}")
    print(f"Rotation Matrix:\n{np.round(T_rot.A, 4)}")
    print(f"Combined Matrix:\n{np.round(T_combined.A, 4)}")
    print(f"Final position: {np.round(T_combined.t, 4)}")
    print(f"Rotation submatrix (orientation):\n{np.round(T_combined.A[:3,:3], 4)}")

    ax = fig.add_subplot(1, 3, idx+1, projection='3d')
    plot_frame_3d(T_origin, ax, label='O', scale=0.4)
    plot_frame_3d(T_combined, ax, label='Final', scale=0.4)
    ax.set_xlim(-1,4); ax.set_ylim(-1,5); ax.set_zlim(-1,6)
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    ax.set_title(f'Trans + Rot{axis} 60°')

plt.suptitle('Task 8 — 3D Translation + Rotation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
### Task 9 — Multiple Rotations

The tool rotates 30° about X, then 45° about Y, then 60° about Z. Since rotations don't commute, we also check the reversed order (Z → Y → X) and compare.

In [ ]:
Rx = SE3.Rx(np.deg2rad(30))
Ry = SE3.Ry(np.deg2rad(45))
Rz = SE3.Rz(np.deg2rad(60))

print("Rx (30°):"); print(np.round(Rx.A, 4))
print("\nRy (45°):"); print(np.round(Ry.A, 4))
print("\nRz (60°):"); print(np.round(Rz.A, 4))

T_xyz = Rx * Ry * Rz          # X → Y → Z
T_zyx = Rz * Ry * Rx          # Z → Y → X (reversed)

print("\nCombined (X→Y→Z):"); print(np.round(T_xyz.A, 4))
print("\nCombined (Z→Y→X):"); print(np.round(T_zyx.A, 4))
print("\nAre they equal?", np.allclose(T_xyz.A, T_zyx.A))

fig = plt.figure(figsize=(9, 4))
T_origin = SE3(0, 0, 0)
for idx, (T, title) in enumerate([(T_xyz, 'X→Y→Z'), (T_zyx, 'Z→Y→X')]):
    ax = fig.add_subplot(1, 2, idx+1, projection='3d')
    plot_frame_3d(T_origin, ax, label='O', scale=0.4)
    plot_frame_3d(T, ax, label='Final', scale=0.4)
    ax.set_xlim(-1,1); ax.set_ylim(-1,1); ax.set_zlim(-1,1)
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    ax.set_title(title)

plt.suptitle('Task 9 — Multiple Rotations', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
### Task 10 — Complete Spatial Transformation

Full sequence: Translate(2,3,1) → Rx(30°) → Ry(45°) → Rz(60°). We also try a different rotation order at the end and compare the results.

In [ ]:
T_t  = SE3(2, 3, 1)
T_rx = SE3.Rx(np.deg2rad(30))
T_ry = SE3.Ry(np.deg2rad(45))
T_rz = SE3.Rz(np.deg2rad(60))

T_seq = T_t * T_rx * T_ry * T_rz          # original order
T_mod = T_t * T_rz * T_ry * T_rx          # modified: rotations reversed

print("Individual matrices:")
for name, T in [('Translation', T_t), ('Rx(30°)', T_rx), ('Ry(45°)', T_ry), ('Rz(60°)', T_rz)]:
    print(f"{name}:\n{np.round(T.A, 4)}")

print("\nFinal Combined Matrix (T→Rx→Ry→Rz):")
print(np.round(T_seq.A, 4))
print(f"Final position: X={T_seq.t[0]:.4f}, Y={T_seq.t[1]:.4f}, Z={T_seq.t[2]:.4f}")
print(f"Orientation matrix:\n{np.round(T_seq.A[:3,:3], 4)}")

print("\nModified Order (T→Rz→Ry→Rx):")
print(np.round(T_mod.A, 4))
print(f"Modified position: X={T_mod.t[0]:.4f}, Y={T_mod.t[1]:.4f}, Z={T_mod.t[2]:.4f}")

fig = plt.figure(figsize=(9, 4))
T_origin = SE3(0, 0, 0)
for idx, (T, title) in enumerate([(T_seq, 'T→Rx→Ry→Rz'), (T_mod, 'T→Rz→Ry→Rx')]):
    ax = fig.add_subplot(1, 2, idx+1, projection='3d')
    plot_frame_3d(T_origin, ax, label='O', scale=0.3)
    plot_frame_3d(T, ax, label='Final', scale=0.3)
    ax.set_xlim(-1,4); ax.set_ylim(-1,5); ax.set_zlim(-1,3)
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    ax.set_title(title)

plt.suptitle('Task 10 — Complete Spatial Transformation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Part C — Application-Based Tasks

---
### Task 11 — Two-Link Planar Robot

Link 1 has length 4 and joint angle 30°. Link 2 has length 3 and joint angle 45° *relative to Link 1*. The overall transformation is T = T1 × T2, and the end-effector position is extracted from the last column.

In [ ]:
L1, theta1 = 4, 30   # link 1
L2, theta2 = 3, 45   # link 2 (angle relative to link 1)

# Transform for link 1: rotate by theta1, then translate along X by L1
T1 = SE2.Rot(np.deg2rad(theta1)) * SE2(L1, 0)
# Transform for link 2: rotate by theta2 (relative), then translate by L2
T2 = SE2.Rot(np.deg2rad(theta2)) * SE2(L2, 0)

T_total = T1 * T2    # end-effector transform w.r.t. world frame

print(f"T1 (Link 1):\n{np.round(T1.A, 4)}")
print(f"\nT2 (Link 2):\n{np.round(T2.A, 4)}")
print(f"\nOverall T = T1 × T2:\n{np.round(T_total.A, 4)}")
print(f"\nEnd-effector position: X={T_total.t[0]:.4f}, Y={T_total.t[1]:.4f}")
angle_ee = np.rad2deg(np.arctan2(T_total.A[1,0], T_total.A[0,0]))
print(f"End-effector orientation: {angle_ee:.4f}°")

# Plot the two-link arm
fig, ax = plt.subplots(figsize=(7, 6))

# joint positions
p0 = np.array([0.0, 0.0])                        # base
p1 = T1.t                                          # elbow
p2 = T_total.t                                     # end-effector

ax.plot([p0[0], p1[0]], [p0[1], p1[1]], 'b-o', lw=3, label='Link 1')
ax.plot([p1[0], p2[0]], [p1[1], p2[1]], 'g-o', lw=3, label='Link 2')
ax.plot(*p2, 'r*', ms=12, label='End-effector')

# draw frames at each joint
plot_frame_2d(SE2(0,0), ax, label='Base', scale=0.4)
plot_frame_2d(T1, ax, label='Elbow', scale=0.4)
plot_frame_2d(T_total, ax, label='EE', scale=0.4)

ax.set_xlim(-2, 8); ax.set_ylim(-2, 8)
ax.set_aspect('equal'); ax.grid(True, ls='--', alpha=0.5)
ax.legend(); ax.set_xlabel('X'); ax.set_ylabel('Y')
ax.set_title(f'Two-Link Planar Robot (θ1={theta1}°, θ2={theta2}°)')
plt.tight_layout()
plt.show()

# vary joint angles
print("\nEffect of changing joint angles:")
for a1, a2 in [(0, 0), (45, 30), (90, -30), (60, 60)]:
    T_test = (SE2.Rot(np.deg2rad(a1)) * SE2(L1, 0)) * (SE2.Rot(np.deg2rad(a2)) * SE2(L2, 0))
    print(f"  θ1={a1}°, θ2={a2}° → EE at ({T_test.t[0]:.3f}, {T_test.t[1]:.3f})")

---
### Task 12 — Two-Link Spatial Robot

Link 1 translates to (0,0,3) and rotates 30° about Z. Link 2 translates by (2,0,0) relative to Link 1's frame and rotates 45° about Y. The overall transform is T = T1 × T2.

In [ ]:
# Link 1: translate (0,0,3) then rotate 30° about Z
T1 = SE3(0, 0, 3) * SE3.Rz(np.deg2rad(30))
# Link 2: translate (2,0,0) relative to Link 1's frame, then rotate 45° about Y
T2 = SE3(2, 0, 0) * SE3.Ry(np.deg2rad(45))

T_total = T1 * T2

print(f"T1 (Link 1):\n{np.round(T1.A, 4)}")
print(f"\nT2 (Link 2):\n{np.round(T2.A, 4)}")
print(f"\nOverall T = T1 × T2:\n{np.round(T_total.A, 4)}")
print(f"\nEnd-effector position: {np.round(T_total.t, 4)}")
print(f"Final orientation (rotation submatrix):\n{np.round(T_total.A[:3,:3], 4)}")

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

T_origin = SE3(0, 0, 0)
p0 = np.array([0., 0., 0.])
p1 = T1.t
p2 = T_total.t

ax.plot(*zip(p0, p1), 'b-o', lw=2, label='Link 1')
ax.plot(*zip(p1, p2), 'g-o', lw=2, label='Link 2')

plot_frame_3d(T_origin, ax, label='Base', scale=0.3)
plot_frame_3d(T1, ax, label='J1', scale=0.3)
plot_frame_3d(T_total, ax, label='EE', scale=0.3)

ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
ax.set_title('Task 12 — Two-Link Spatial Robot')
ax.legend()
plt.tight_layout()
plt.show()

print("\nEffect of changing angles:")
for az, ay in [(0, 0), (45, 30), (90, 60), (-30, 45)]:
    T1t = SE3(0,0,3) * SE3.Rz(np.deg2rad(az))
    T2t = SE3(2,0,0) * SE3.Ry(np.deg2rad(ay))
    Tt  = T1t * T2t
    print(f"  Rz={az}°, Ry={ay}° → EE at {np.round(Tt.t, 3)}")

---
## Part D — Challenge Task

---
### Task 13 — Complete Transformation Sequence

The robot performs a full sequence — four planar steps followed by five spatial steps. Since SE2 and SE3 live in different spaces, we embed the planar part into 3D (using the XY plane, Z=0) for a unified analysis. We also compare with a modified transformation order.

In [ ]:
# ── Planar Transformation Sequence (SE2) ──
P1 = SE2(2, 3)                          # translate (2,3)
P2 = SE2.Rot(np.deg2rad(45))            # rotate 45°
P3 = SE2(4, 0)                          # translate (4,0)
P4 = SE2.Rot(np.deg2rad(30))            # rotate 30°

T_planar = P1 * P2 * P3 * P4

print("=== Planar Sequence ===")
for name, T in [('P1: Trans(2,3)', P1), ('P2: Rot(45°)', P2),
                ('P3: Trans(4,0)', P3), ('P4: Rot(30°)', P4)]:
    print(f"{name}:\n{np.round(T.A, 4)}")

print(f"\nFinal Planar Matrix:\n{np.round(T_planar.A, 4)}")
print(f"Final planar position: X={T_planar.t[0]:.4f}, Y={T_planar.t[1]:.4f}")
angle_planar = np.rad2deg(np.arctan2(T_planar.A[1,0], T_planar.A[0,0]))
print(f"Final planar orientation: {angle_planar:.4f}°")

In [ ]:
# ── Spatial Transformation Sequence (SE3) ──
S1 = SE3(2, 3, 1)                        # translate (2,3,1)
S2 = SE3.Rx(np.deg2rad(30))              # rotate 30° about X
S3 = SE3.Ry(np.deg2rad(45))              # rotate 45° about Y
S4 = SE3(1, 2, 3)                        # translate (1,2,3)
S5 = SE3.Rz(np.deg2rad(60))              # rotate 60° about Z

T_spatial = S1 * S2 * S3 * S4 * S5      # original order

print("\n=== Spatial Sequence ===")
for name, T in [('S1: Trans(2,3,1)', S1), ('S2: Rx(30°)', S2),
                ('S3: Ry(45°)', S3),      ('S4: Trans(1,2,3)', S4),
                ('S5: Rz(60°)', S5)]:
    print(f"{name}:\n{np.round(T.A, 4)}")

print(f"\nFinal Spatial Matrix:\n{np.round(T_spatial.A, 4)}")
print(f"Final position: X={T_spatial.t[0]:.4f}, Y={T_spatial.t[1]:.4f}, Z={T_spatial.t[2]:.4f}")
print(f"Orientation matrix:\n{np.round(T_spatial.A[:3,:3], 4)}")

In [ ]:
# ── Modified order: swap S2 and S3 (Rx and Ry swapped), and S4/S5 swapped ──
T_modified = S1 * S3 * S2 * S5 * S4

print("Modified order (S1→S3→S2→S5→S4):")
print(np.round(T_modified.A, 4))
print(f"Modified position: {np.round(T_modified.t, 4)}")

print("\nDifference between original and modified matrices:")
print(np.round(T_spatial.A - T_modified.A, 4))
print("\nAre they the same?", np.allclose(T_spatial.A, T_modified.A))

# Visualize both spatial results
fig = plt.figure(figsize=(10, 5))
T_origin = SE3(0, 0, 0)
for idx, (T, title) in enumerate([(T_spatial, 'Original Order'), (T_modified, 'Modified Order')]):
    ax = fig.add_subplot(1, 2, idx+1, projection='3d')
    plot_frame_3d(T_origin, ax, label='O', scale=0.3)
    plot_frame_3d(T, ax, label='Final', scale=0.3)
    ax.set_xlim(-1,5); ax.set_ylim(-1,7); ax.set_zlim(-1,5)
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    ax.set_title(title)

plt.suptitle('Task 13 — Complete Transformation Sequence', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()